## ✅ 오즈비, 카이제곱, Cramer's V 계산(3분화 버전 파일로 계산)
------------------------------------------------------------
### 목표
 1) stats/pivots.py: long df -> (필터/조건 반영) 교차표(tab/pivot) 생성만 담당: stats.pivots
 2) 교차표로부터 로그오즈비 계산만 담당: from stats.logodds
 3) 교차표로부터 카이제곱/크래머V 계산만 담당: from stats.chi2_utils


### 두 방식으로 돌아감.
 - (A) context × outcome (multi-level outcome) : tab (RxC)

    : make_context_outcome_tab, logodds_context_by_outcome_from_tab, chi2_overall_from_tab
    

 - (B) unit × context × binary-outcome : pivot (unit별 2xK + unit×context 2x2 log-odds)

    : make_unit_context_binary_pivot, logodds_unit_by_context_from_pivot, chi2_by_unit_from_pivot


In [ ]:
# ------------------------------------------------------------------
# ✅ 사용 패턴 예시 (너의 기존 함수들을 "조합"으로 대체)
# ------------------------------------------------------------------
"""
# (A) context × outcome (multi-level)
from stats.pivots import make_context_outcome_tab
from stats.logodds import logodds_context_by_outcome_from_tab
from stats.chi2_utils import chi2_overall_from_tab

tab, ctx_vals, out_vals, mode = make_context_outcome_tab(
    df,
    context_col="category",
    outcome_col="EP_T",
    count_mode="weight",
    weight_col="ID",
    filters={...},
    context_top_n=10,
    outcome_top_n=30,
)

result_df = logodds_context_by_outcome_from_tab(
    tab,
    context_col_name="category",
    outcome_col_name="outcome_level",#tab의 컬럼 이름이 outcome_level이므로 이 이름을 기본값으로 설정함.
    alpha=0.5,
    add_alpha_if_zero=True,
    count_mode=mode,
)

chi2_df = chi2_overall_from_tab(tab, count_mode=mode)


# (B) unit × context × binary outcome (동사별 장르차)
from stats.pivots import make_unit_context_binary_pivot
from stats.logodds import logodds_unit_by_context_from_pivot
from stats.chi2_utils import chi2_by_unit_from_pivot

pivot, unit_vals, ctx_vals, mode = make_unit_context_binary_pivot(
    df,
    unit_col="V_No",
    context_col="category",
    outcome_col="EP_T",
    positive_fn=default_binary_parser, # True/False 변환 함수 예) positive_fn = lambda x: "었" in str(x)
    count_mode="weight",
    weight_col="ID",
    filters={...},
    unit_top_n=1000,
)

logodds_df = logodds_unit_by_context_from_pivot(
    pivot,
    unit_col_name="V_No",
    context_col_name="category",
    alpha=0.5,
    add_alpha_if_zero=True,
    count_mode=mode,
)

chi2_unit_df = chi2_by_unit_from_pivot(
    pivot,
    unit_col_name="V_No",
    count_mode=mode,
)
"""


'\n# (A) context × outcome (multi-level)\nfrom stats.pivots import make_context_outcome_tab\nfrom stats.logodds import logodds_context_by_outcome_from_tab\nfrom stats.chi2_utils import chi2_overall_from_tab\n\ntab, ctx_vals, out_vals, mode = make_context_outcome_tab(\n    df,\n    context_col="category",\n    outcome_col="EP_T",\n    count_mode="weight",\n    weight_col="ID",\n    filters={...},\n    context_top_n=10,\n    outcome_top_n=30,\n)\n\nresult_df = logodds_context_by_outcome_from_tab(\n    tab,\n    context_col_name="category",\n    outcome_col_name="outcome_level",#tab의 컬럼 이름이 outcome_level이므로 이 이름을 기본값으로 설정함.\n    alpha=0.5,\n    add_alpha_if_zero=True,\n    count_mode=mode,\n)\n\nchi2_df = chi2_overall_from_tab(tab, count_mode=mode)\n\n\n# (B) unit × context × binary outcome (동사별 장르차)\nfrom stats.pivots import make_unit_context_binary_pivot\nfrom stats.logodds import logodds_unit_by_context_from_pivot\nfrom stats.chi2_utils import chi2_by_unit_from_pivot\n\npivot, unit_val

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0]
sys.path.append(str(ROOT))

from stats.pivots import make_context_outcome_tab, make_unit_context_binary_pivot #이 함수들은 피벗 테이블을 만들어주는 함수로, logodds나 chi2 계산 전에 데이터를 준비하는 단계에서 사용됨.
from stats.logodds import logodds_context_by_outcome_from_tab, logodds_unit_by_context_from_pivot #이 함수들은 피벗 테이블을 입력으로 받아서 logodds 계산을 수행하는 함수로, 실제로 logodds 계산을 하는 단계에서 사용됨.
from stats.chi2_utils import chi2_overall_from_tab, chi2_by_unit_from_pivot #이 함수들은 피벗 테이블을 입력으로 받아서 카이제곱 검정을 수행하는 함수로, 실제로 chi2 계산을 하는 단계에서 사용됨.


In [89]:
from datetime import datetime
import pandas as pd
import numpy as np

CSV_PATH = r"..\csv\통계용\document_구분1_20260322_23-56.csv"
COUNT_MODE = "weight" #가중치 파일
WEIGHT_COL="count" #입력 파일이 이미 가중치가 적용된 형태이므로, 가중치 컬럼을 지정하여 "개수" 대신 "가중치 합계"로 계산하도록 함.

df = pd.read_csv(CSV_PATH)

print(f"CSV 파일 로드 완료: {len(df):,}, now: {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")
df.columns

CSV 파일 로드 완료: 264,768, now: 2026.03.23_00:10:52


Index(['Unnamed: 0', 'file_id', 'docu_id', 'speaker', 'quote_type',
       'sent_has_E', 'sentence_f_EN_No', 'sentence_f_EN_label',
       'sentence_f_EN_in_quote', 'sentence_f_EP_form', 'sentence_f_EP_TT',
       'sentence_f_EP_T', 'sentence_f_EP_M', 'count'],
      dtype='object')

In [78]:
df[df['VX0_order']==-1]['VX0_No'].value_counts()

VX0_No
-1    876011
Name: count, dtype: int64

In [86]:
print(f"전체: {df['count'].count():,} 행\n")
print(f"EN_label 개수 : {df['EN_label'].count():,}")
print(f"EN_label Null: {df[df['EN_label'].isna()]['count'].count():,}\n")
print(f"V_label 개수 : {df['V_label'].count():,}")
print(f"V_label Null: {df[df['V_label'].isna()]['count'].count():,}\n")

print(f"sentence 수:           {df[df['VX0_order']==-1]['count'].count():,}")
print(f"sentence 수(body만):   {df[(df['VX0_order']==-1)&(df['speaker']=='body')]['count'].count():,}")
print(f"non_quote_sentence 수: {df[(df['VX0_order']==-1)&(df['sentence_sent_end_V_in_quote']==False)]['count'].count():,}")
print(f"non_quote & body:      {df[(df['VX0_order']==-1)&(df['speaker']=='body')
                                   &(df['sentence_sent_end_V_in_quote']==False)]['count'].count():,}\n")

print(f"첫문장 수:               {df[(df['VX0_order']==-1)&(df['has_prev_sentence']==False)&(df['speaker']=='body')]['count'].count():,}")
print(f"마지막 문장 수:          {df[(df['VX0_order']==-1)&(df['has_next_sentence']==False)&(df['speaker']=='body')]['count'].count():,}")


전체: 1,224,091 행

EN_label 개수 : 1,223,838
EN_label Null: 253

V_label 개수 : 1,221,271
V_label Null: 2,820

sentence 수:           876,011
sentence 수(body만):   859,618
non_quote_sentence 수: 743,392
non_quote & body:      728,482

첫문장 수:               44
마지막 문장 수:          23,243


In [95]:
# (A) context × outcome (multi-level)
from datetime import datetime
start_time = datetime.now()
print(f"시작: {start_time.strftime('%Y.%m.%d_%H:%M:%S')}")

MOAD = "A"
CONTEXT_COL="file_id"
OUTCOME_COL="sentence_f_EP_T"
FILTERS_SENTENCE_T = {
    "speaker": "body",
    "sent_has_E": True,
    "sentence_f_EN_in_quote": False,
}
FILTERS_NEWS_1101 = {
    #'매체': '신문',
    #"f_EN_label": "EF",
    #'f_EN_No': 1101,
    "speaker": "body",
    #'sen_count_has_E_not_quote': lambda x: x > 9,
    #"분석대상": "대상",
}

FILTER_T_SWITCH = {
    "speaker": "body",
    "VX0_No": -1,     
    'has_prev_sentence': True
}
if False:# True: #True, False: #필터 예시 - 특정 조건을 만족하는 행만 분석에 포함시키고 싶을 때, FILTERS_NEWS_1101 딕셔너리에 조건을 추가할 수 있음. 예를 들어, 'vx0_No' 컬럼이 -1인 행만 포함시키고 싶다면, 다음과 같이 추가할 수 있음.
    FILTER_V0 = {"VX0_No": -1,     
                }
    FILTERS_NEWS_1101.update(FILTER_V0)
    

tab, ctx_vals, out_vals, mode = make_context_outcome_tab(
    df,
    context_col=CONTEXT_COL,
    outcome_col=OUTCOME_COL,
    count_mode=COUNT_MODE,
    weight_col=WEIGHT_COL,
    filters=FILTERS_SENTENCE_T,
    #context_top_n=30,
    #outcome_top_n=30,
)

result_df = logodds_context_by_outcome_from_tab(
    tab,
    context_col_name=CONTEXT_COL,
    alpha=0.5,
    add_alpha_if_zero=True,
    count_mode=mode,
)

chi2_df = chi2_overall_from_tab(tab, count_mode=mode)
print(f"[OK] made pivot, logodds, chi2: {OUTCOME_COL}_by_{CONTEXT_COL}, now: {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")


시작: 2026.03.23_01:21:23
[OK] made pivot, logodds, chi2: sentence_f_EP_T_by_file_id, now: 2026.03.23_01:21:23


In [96]:
# =========================================================
# 4. save to file
# =========================================================

#---- file name settings ----  
SAVE_DIR = Path("..") / "csv"/"결과값"/"tense"
timestamp = start_time.strftime('%Y%m%d_%H-%M')

out_file_oz = SAVE_DIR / f'{MOAD}_oz_{OUTCOME_COL}_by_{CONTEXT_COL}_{timestamp}.csv'
out_file_chi2 = SAVE_DIR / f'{MOAD}_chi2_{OUTCOME_COL}_by_{CONTEXT_COL}_{timestamp}.csv'
out_pivot = SAVE_DIR / f'{MOAD}_pivot_{OUTCOME_COL}_by_{CONTEXT_COL}_{timestamp}.csv'

print(f"***저장합니다.:    {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}***")
tab.to_csv(out_pivot,index = False, encoding="utf-8-sig")
print(f"Output file for pivot table: {out_pivot}")

result_df.to_csv(out_file_oz, index=False, encoding="utf-8-sig")
print(f"[OK] saved logodds_{out_file_oz}")

if chi2_df is not None:
    chi2_df.to_csv(out_file_chi2, index=False, encoding="utf-8-sig")
    print(f"[OK] saved chi2_{out_file_chi2}")

***저장합니다.:    2026.03.23_01:21:27***
Output file for pivot table: ..\csv\결과값\tense\A_pivot_sentence_f_EP_T_by_file_id_20260323_01-21.csv
[OK] saved logodds_..\csv\결과값\tense\A_oz_sentence_f_EP_T_by_file_id_20260323_01-21.csv
[OK] saved chi2_..\csv\결과값\tense\A_chi2_sentence_f_EP_T_by_file_id_20260323_01-21.csv


In [66]:
# (B) unit × context × binary outcome 
MOAD = "B"

FILTERS_NEWS_1101 = {
    '매체': '신문',
    "f_EN_label": "EF",
    'f_EN_No': 1101,
    "speaker": "body",
    'sen_count_has_E_not_quote': lambda x: x > 9,
    "분석대상": "대상",
    'has_prev_sentence': True, #이 조건을 추가하여, 이전 문장이 존재하는 경우만 분석에 포함시키도록 함. 이렇게 하면, presentence_f_EP_T 컬럼이 NaN인 경우(즉, 이전 문장이 없는 경우)는 분석에서 제외됨.
    'EN_label': "ETM"
}
FILTER_T_SWITCH = {
    "speaker": "body",
    "VX0_No": -1,     
    'has_prev_sentence': True
}
UNIT_COL="category"
CONTEXT_COL="sentence_f_EP_T"
OUTCOME_COL="presentence_f_EP_T"

from stats.pivots import make_unit_context_binary_pivot
from stats.logodds import logodds_unit_by_context_from_pivot
from stats.chi2_utils import chi2_by_unit_from_pivot

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

pivot, unit_vals, ctx_vals, mode = make_unit_context_binary_pivot(
    df,
    unit_col=UNIT_COL,
    context_col=CONTEXT_COL,
    outcome_col=OUTCOME_COL,
    #positive_fn=default_binary_parser,
    count_mode=COUNT_MODE,
    weight_col=WEIGHT_COL,
    filters=FILTER_T_SWITCH,
    #unit_top_n=1000,
)
print(f"[OK] made pivot: {UNIT_COL}X{CONTEXT_COL}_{OUTCOME_COL}, 실행시간: {timestamp}")


[OK] made pivot: categoryXsentence_f_EP_T_presentence_f_EP_T, 실행시간: 2026-03-22_07-58


In [67]:
logodds_df = logodds_unit_by_context_from_pivot(
    pivot,
    unit_col_name=UNIT_COL,
    context_col_name=CONTEXT_COL,
    alpha=0.5,
    add_alpha_if_zero=True,
    count_mode=mode,
)

chi2_unit_df = chi2_by_unit_from_pivot(
    pivot,
    unit_col_name=UNIT_COL,
    count_mode=mode,
)
# =========================================================
# 4. save to file
# =========================================================

#---- file name settings ----  
SAVE_DIR = Path("..") / "csv"/"결과값"/"tense"
out_pivot = SAVE_DIR / f'{MOAD}_pivot_{UNIT_COL}_{OUTCOME_COL}_by_{CONTEXT_COL}_{timestamp}.csv'
out_file_oz = SAVE_DIR / f'{MOAD}_oz_{UNIT_COL}_{OUTCOME_COL}_by_{CONTEXT_COL}_{timestamp}.csv'
out_file_chi2 = SAVE_DIR / f'{MOAD}_chi2_{UNIT_COL}_{OUTCOME_COL}_by_{CONTEXT_COL}_{timestamp}.csv'

print(f"***저장합니다.:    {datetime.now()}***")
pivot_reset = pivot.reset_index()
pivot_reset.to_csv(out_pivot, index=False, encoding="utf-8-sig")
print(f"[OK] saved pivot_{out_pivot}")
logodds_df.to_csv(out_file_oz, index=False, encoding="utf-8-sig")
print(f"[OK] saved logodds_{out_file_oz}")

if chi2_unit_df is not None:
    chi2_unit_df.to_csv(out_file_chi2, index=False, encoding="utf-8-sig")
    print(f"[OK] saved chi2_{out_file_chi2}")

***저장합니다.:    2026-03-22 07:58:21.675807***
[OK] saved pivot_..\csv\결과값\tense\B_pivot_category_presentence_f_EP_T_by_sentence_f_EP_T_2026-03-22_07-58.csv
[OK] saved logodds_..\csv\결과값\tense\B_oz_category_presentence_f_EP_T_by_sentence_f_EP_T_2026-03-22_07-58.csv
[OK] saved chi2_..\csv\결과값\tense\B_chi2_category_presentence_f_EP_T_by_sentence_f_EP_T_2026-03-22_07-58.csv
